# Multi-Seed EgyBERT Comparison

Runs the matched `L1+L2+L3` `EgyBERT` comparison across three fixed seeds: `42`, `123`, and `456`.

This notebook mirrors the structure of `kaggle_multi_seed_training.ipynb`, but limits the run set to the single `EgyBERT` comparison configuration.

In [ ]:
# Clone the repository
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding the cloud's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml -q

In [ ]:
# Link the processed dataset
!rm -rf data/processed && mkdir -p data/processed
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/
!ls data/processed
# Expected: label_encoders.pkl  test.csv  train.csv  val.csv

In [ ]:
!nvidia-smi

In [ ]:
# Download EgyBERT fully to local disk before training.
from huggingface_hub import snapshot_download

print('Downloading EgyBERT to local disk...')
EGYBERT_PATH = snapshot_download(
    'faisalq/EgyBERT',
    local_dir='/kaggle/working/model_cache/egybert'
)
print('EgyBERT ready at:', EGYBERT_PATH)

In [ ]:
from pathlib import Path

METRICS_DIR = Path('results/metrics/multi_seed')
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]

CONFIGS = [
    {
        'name': 'egybert_l1l2l3',
        'model': EGYBERT_PATH,
        'flags': '--tasks l1 l2 l3',
        'task_label': 'l1_l2_l3',
        'checkpoint_prefix': 'egybert',
        'epochs': 10,
    }
]

print('Total runs:', len(CONFIGS) * len(SEEDS))
for cfg in CONFIGS:
    for seed in SEEDS:
        print(' ', cfg['name'], 'seed=' + str(seed))

In [ ]:
import json, time

ipy = get_ipython()
run_log = []

for cfg in CONFIGS:
    for seed in SEEDS:
        run_key = cfg['name'] + '_seed' + str(seed)
        out_metrics_path = METRICS_DIR / (run_key + '_metrics.json')

        if out_metrics_path.exists():
            print('[SKIP]', run_key, '-- already exists')
            continue

        seed_dir = 'models/seed_' + str(seed)
        checkpoint_dir = Path(seed_dir) / (cfg['checkpoint_prefix'] + '_' + cfg['task_label'] + '_best')

        parts = [
            'python scripts/train.py',
            cfg['flags'],
            '--model', cfg['model'],
            '--epochs', str(cfg['epochs']),
            '--seed', str(seed),
            '--output-dir', seed_dir,
            '--data-dir data/processed',
        ]
        cmd = ' '.join(parts)

        print('')
        print('[RUN]', run_key)
        print('  CMD:', cmd)
        print('  Checkpoint will be:', checkpoint_dir)
        t0 = time.time()
        ipy.system(cmd)
        elapsed = time.time() - t0
        print('  Time: %.1f min' % (elapsed / 60,))

        test_metrics_file = checkpoint_dir / 'test_metrics.json'
        if test_metrics_file.exists():
            with open(test_metrics_file) as fp:
                test_metrics = json.load(fp)
            result = {'config': cfg['name'], 'seed': seed, 'metrics': test_metrics}
            with open(out_metrics_path, 'w') as fp:
                json.dump(result, fp, indent=2)
            print('  Saved:', out_metrics_path)
            run_log.append({'run': run_key, 'status': 'ok', 'time_min': elapsed / 60})
        else:
            print('  WARNING: test_metrics.json not found at', test_metrics_file)
            run_log.append({'run': run_key, 'status': 'no_metrics', 'time_min': elapsed / 60})

print('')
print('=== Run log ===')
for r in run_log:
    print(' ', r['run'] + ':', r['status'], '(%.1f min)' % r['time_min'])

In [ ]:
# Aggregate results
!python scripts/aggregate_multi_seed.py --metrics-dir results/metrics/multi_seed

In [ ]:
# View only the EgyBERT summary row
import json
with open('results/metrics/multi_seed_summary.json') as f:
    summary = json.load(f)

data = summary['summary']['egybert_l1l2l3']
print('egybert_l1l2l3 (seeds: %s):' % data['seeds'])
for task, stats in data['tasks'].items():
    print(f'  {task}: {stats["mean"]*100:.2f} ± {stats["std"]*100:.2f} (F1%)')

In [ ]:
# Package the EgyBERT multi-seed metrics for download
!zip -r egybert_multi_seed_results.zip results/metrics/multi_seed/egybert_l1l2l3_seed*_metrics.json results/metrics/multi_seed_summary.json
print('Created egybert_multi_seed_results.zip - download from Kaggle output')